# CS 521 HW3, Problem 1: Standard and Adversarial Training


# Set up for dataset and model


In [13]:
from pathlib import Path
import gdown


def find_hw3_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / 'q3').is_dir() and (candidate / 'pyproject.toml').is_file():
            return candidate
        if (candidate / 'hw3' / 'q3').is_dir():
            return candidate / 'hw3'
    # A standalone Colab upload uses the runtime's current directory.
    return cwd


HW3_ROOT = find_hw3_root()
DATA_DIR = HW3_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
FILES = [
    ('1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe', 'cifar10_train_10k.pt'),
    ('1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA', 'cifar10_test_1k.pt'),
]
for file_id, filename in FILES:
    destination = DATA_DIR / filename
    if not destination.is_file():
        partial = destination.with_suffix('.pt.part')
        downloaded = gdown.download(id=file_id, output=str(partial), quiet=False, use_cookies=False)
        if downloaded is None:
            raise RuntimeError(f'Download failed: {filename}; place the assignment file in {DATA_DIR}')
        partial.replace(destination)
print('Shared data:', DATA_DIR)

Shared data: /Users/zhiyuwu2/uiuc/CCS521FA26HW/hw3/data


In [14]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import os

# HW3_DEVICE=cpu/cuda/mps can override automatic device selection.
automatic_device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device = torch.device(os.environ.get("HW3_DEVICE", automatic_device))
print("device:", device)

EPS = {"linf": 8 / 255}
EPSILONS = (2 / 255, 4 / 255, 8 / 255, 12 / 255, 16 / 255)

device: mps


In [15]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(DATA_DIR / path, map_location="cpu")
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [16]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


# Implement the Attacks


In [17]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.empty_like(x).uniform_(-eps, eps) if rand_init else torch.zeros_like(x)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = (delta + alpha * grad.sign()).clamp(-eps, eps)
        delta = clip_box(x, delta).detach()
    return (x + delta).detach()


In [18]:
ATTACKS = {"linf": pgd_linf}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Clean, FGSM, and PGD Accuracy


In [19]:
def test_model(model, eps=8 / 255, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", "fgsm", "pgd")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        fgsm = clean.clone()
        robust = clean.clone()
        if eps > 0:
            x_adv = pgd_linf(model, x, y, eps, eps, 1, rand_init=False)
            with torch.no_grad():
                fgsm &= model(x_adv).argmax(1) == y
            for _ in range(restarts):
                x_adv = pgd_linf(model, x, y, eps, 2.5 * eps / steps, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
        correct["clean"] += clean.sum().item()
        correct["fgsm"] += fgsm.sum().item()
        correct["pgd"] += robust.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print(f"eps {eps * 255:g}/255  " + "  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc


# Adversarial Training


In [20]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Standard Training


In [21]:
def standard_loss(model, x, y):
    model.train()
    return F.cross_entropy(model(x), y)


In [22]:
torch.manual_seed(0)
model_standard = adv_train(make_model(), standard_loss, epochs=5)
results_standard = {}
for eps in EPSILONS:
    results_standard[eps] = test_model(model_standard, eps=eps)


epoch 1: loss 1.890 (2 s)
epoch 2: loss 1.531 (5 s)
epoch 3: loss 1.350 (7 s)
epoch 4: loss 1.217 (10 s)
epoch 5: loss 1.128 (12 s)
eps 2/255  clean 58.9  fgsm 27.5  pgd 22.7   (15 s)
eps 4/255  clean 58.9  fgsm 12.5  pgd 5.8   (15 s)
eps 8/255  clean 58.9  fgsm 3.1  pgd 0.4   (15 s)
eps 12/255  clean 58.9  fgsm 1.2  pgd 0.0   (15 s)
eps 16/255  clean 58.9  fgsm 0.7  pgd 0.0   (15 s)


# Single-Norm Adversarial Training


In [23]:
def single_norm_loss(model, x, y, norm):
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)

In [24]:
torch.manual_seed(0)
model_linf = adv_train(make_model(), single_norm_loss, epochs=5, norm="linf")
results_linf = {}
for eps in EPSILONS:
    results_linf[eps] = test_model(model_linf, eps=eps)


epoch 1: loss 2.190 (16 s)
epoch 2: loss 2.097 (33 s)
epoch 3: loss 2.052 (49 s)
epoch 4: loss 2.028 (66 s)
epoch 5: loss 2.015 (82 s)
eps 2/255  clean 38.1  fgsm 33.8  pgd 33.8   (15 s)
eps 4/255  clean 38.1  fgsm 30.1  pgd 30.1   (15 s)
eps 8/255  clean 38.1  fgsm 23.6  pgd 23.1   (15 s)
eps 12/255  clean 38.1  fgsm 20.0  pgd 18.1   (15 s)
eps 16/255  clean 38.1  fgsm 15.0  pgd 12.6   (15 s)
